# Deep Q-Network on LunarLander

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

DQN parameterises the action-value function $Q(s, a)$ with a neural net, trains it to satisfy the Bellman equation using a *target network* to stabilise the target, and reuses past experience from a *replay buffer* to break correlations.


## Mathematical Formulation

$$\mathcal{L}(\theta) = \mathbb{E}_{(s, a, r, s')}\!\left[\big(r + \gamma\,\max_{a'} Q_{\theta^-}(s', a') - Q_\theta(s, a)\big)^2\right]$$

$\theta^-$ are the slowly-updated target-net weights.


## Implementation


In [ ]:
import random, collections
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class QNet(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, 128), nn.ReLU(),
                                 nn.Linear(128, 128), nn.ReLU(),
                                 nn.Linear(128, n_actions))
    def forward(self, x): return self.net(x)

class Replay:
    def __init__(self, cap=10000):
        self.buf = collections.deque(maxlen=cap)
    def push(self, *t): self.buf.append(t)
    def sample(self, n):
        batch = random.sample(self.buf, n)
        s, a, r, s2, d = zip(*batch)
        return (torch.tensor(s, dtype=torch.float32),
                torch.tensor(a),
                torch.tensor(r, dtype=torch.float32),
                torch.tensor(s2, dtype=torch.float32),
                torch.tensor(d, dtype=torch.float32))
    def __len__(self): return len(self.buf)


## Experiment


In [ ]:
env = gym.make('LunarLander-v2') if 'LunarLander-v2' in [s.id for s in gym.envs.registry.values()] else gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
online = QNet(obs_dim, n_actions); target = QNet(obs_dim, n_actions)
target.load_state_dict(online.state_dict())
opt = torch.optim.Adam(online.parameters(), lr=1e-3)
buf = Replay()

eps = 1.0
for ep in range(80):
    obs, _ = env.reset(seed=ep)
    total = 0; done = False
    while not done:
        if random.random() < eps:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = online(torch.tensor(obs, dtype=torch.float32)).argmax().item()
        obs2, r, term, trunc, _ = env.step(a)
        buf.push(obs, a, r, obs2, float(term or trunc))
        obs, total, done = obs2, total + r, term or trunc
        if len(buf) >= 64:
            s, ac, rw, s2, d = buf.sample(64)
            q = online(s).gather(1, ac.long().unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                target_q = rw + 0.99 * (1 - d) * target(s2).max(1).values
            loss = F.mse_loss(q, target_q)
            opt.zero_grad(); loss.backward(); opt.step()
    eps = max(0.05, eps * 0.97)
    if ep % 10 == 0:
        target.load_state_dict(online.state_dict())
        print(f'episode {ep:3d}  return {total:.1f}  eps {eps:.2f}')


## Discussion

- Double DQN, dueling heads, prioritised replay are easy upgrades and matter on Atari-like tasks.
- LunarLander needs box2d; if it isn't installed fall back to CartPole — the algorithm is identical.
- Adjust the target-net sync frequency: hard copy every $K$ steps or use Polyak averaging.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
